# production manifest hash validation v1

目标：验证 production health check 不只比较 registry current 中保存的 manifest hash，也会从 manifest 文件内容重算 hash，识别 production manifest 被发布后篡改或漂移的情况。

边界：notebook 只编排证据和临时篡改副本；稳定 hash 逻辑必须沉到 `src.core`，健康检查逻辑必须沉到 `src.schedule`。


In [1]:
from pathlib import Path
import json
import os
import shutil
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import production_manifest_content_hash
from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

CARD = Path('research/production_manifest_hash_validation_v1')
CARD.mkdir(parents=True, exist_ok=True)
REAL_REGISTRY = Path('production/production_registry.json')
STRATEGY_ID = 'paper_sim_replay_v1'
events = []
CARD


PosixPath('research/production_manifest_hash_validation_v1')

In [2]:
healthy_config = ProductionHealthCheckConfig(
    name='research_production_manifest_hash_healthy_v1',
    registry_path=REAL_REGISTRY,
    strategy_id=STRATEGY_ID,
    output_path=CARD / 'healthy_health_export.json',
    notify_details=True,
)
healthy_result = run_production_health_check_once(healthy_config, lock_dir=CARD / 'locks', notify=events.append)
healthy_export = json.loads((CARD / 'healthy_health_export.json').read_text(encoding='utf-8'))
healthy_export


{'schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'registry_path': 'production/production_registry.json',
 'manifest_path': 'production/paper_sim_replay_v1/production_manifest.json',
 'manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'registry_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'computed_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'manifest_hash_matches_registry': True,
 'manifest_hash_matches_content': True,
 'artifact_id': 'paper_sim_replay_v1',
 'artifact_type': 'production_strategy',
 'mode': 'paper',
 'status': 'paper_replay_completed',
 'live_broker_enabled': False,
 'published_at': '2026-06-09T21:30:00Z',
 'start': '2026-01-02',
 'end': '2026-06-09',
 'required_path_count': 7,
 'checked_path_count': 7,
 'missing_required_paths': [],
 'resolved_paths': {'notebook':

In [3]:
registry = json.loads(REAL_REGISTRY.read_text(encoding='utf-8'))
current = registry['strategies'][STRATEGY_ID]['current']
source_manifest = Path(current['manifest_path'])
tamper_dir = CARD / 'tampered_case'
if tamper_dir.exists():
    shutil.rmtree(tamper_dir)
tamper_dir.mkdir(parents=True, exist_ok=True)
tampered_manifest_path = tamper_dir / 'production_manifest.json'
tampered_registry_path = tamper_dir / 'production_registry.json'

tampered_manifest = json.loads(source_manifest.read_text(encoding='utf-8'))
tampered_manifest['metadata'] = dict(tampered_manifest.get('metadata', {}))
tampered_manifest['metadata']['status'] = 'paper_replay_changed_after_publish'
tampered_manifest_path.write_text(json.dumps(tampered_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

tampered_registry = json.loads(json.dumps(registry))
tampered_registry['strategies'][STRATEGY_ID]['current']['manifest_path'] = str(tampered_manifest_path)
tampered_registry_path.write_text(json.dumps(tampered_registry, ensure_ascii=False, indent=2), encoding='utf-8')

stored_hash = str(tampered_manifest['manifest_hash'])
computed_hash = production_manifest_content_hash(tampered_manifest)
{'stored_hash': stored_hash, 'computed_hash': computed_hash, 'hash_changed': stored_hash != computed_hash}


{'stored_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'computed_hash': '46e1b4b612e7f68d583edb593eb9f00c273b159b76263ac5ced87f15ad65fc64',
 'hash_changed': True}

In [4]:
tampered_events = []
tampered_config = ProductionHealthCheckConfig(
    name='research_production_manifest_hash_tampered_v1',
    registry_path=tampered_registry_path,
    strategy_id=STRATEGY_ID,
    output_path=CARD / 'tampered_health_export.json',
    required_paths=(),
    notify_details=True,
)
tampered_result = run_production_health_check_once(tampered_config, lock_dir=CARD / 'locks', notify=tampered_events.append)
tampered_export = json.loads((CARD / 'tampered_health_export.json').read_text(encoding='utf-8'))
tampered_export


{'schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'manifest_hash_mismatch',
 'is_healthy': False,
 'registry_path': 'research/production_manifest_hash_validation_v1/tampered_case/production_registry.json',
 'manifest_path': 'research/production_manifest_hash_validation_v1/tampered_case/production_manifest.json',
 'manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'registry_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'computed_manifest_hash': '46e1b4b612e7f68d583edb593eb9f00c273b159b76263ac5ced87f15ad65fc64',
 'manifest_hash_matches_registry': True,
 'manifest_hash_matches_content': False,
 'artifact_id': 'paper_sim_replay_v1',
 'artifact_type': 'production_strategy',
 'mode': 'paper',
 'status': 'paper_replay_changed_after_publish',
 'live_broker_enabled': False,
 'published_at': '2026-06-09T21:30:00Z',
 'start': '2026-01-02',
 'end': '2026-06-09',
 're

In [5]:
summary = {
    'schema_version': 'research.production_manifest_hash_validation.v1',
    'strategy_id': STRATEGY_ID,
    'healthy_status': healthy_result.status,
    'healthy_health_status': healthy_export['health_status'],
    'healthy_is_healthy': healthy_export['is_healthy'],
    'healthy_manifest_hash_matches_registry': healthy_export['manifest_hash_matches_registry'],
    'healthy_manifest_hash_matches_content': healthy_export['manifest_hash_matches_content'],
    'healthy_missing_required_paths': healthy_export['missing_required_paths'],
    'healthy_manifest_hash': healthy_export['manifest_hash'],
    'healthy_computed_manifest_hash': healthy_export['computed_manifest_hash'],
    'tampered_status': tampered_result.status,
    'tampered_health_status': tampered_export['health_status'],
    'tampered_is_healthy': tampered_export['is_healthy'],
    'tampered_manifest_hash_matches_registry': tampered_export['manifest_hash_matches_registry'],
    'tampered_manifest_hash_matches_content': tampered_export['manifest_hash_matches_content'],
    'tampered_manifest_hash': tampered_export['manifest_hash'],
    'tampered_computed_manifest_hash': tampered_export['computed_manifest_hash'],
    'healthy_notification': events[0] if events else '',
    'tampered_notification': tampered_events[0] if tampered_events else '',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

assert healthy_result.status == 'success'
assert healthy_export['health_status'] == 'healthy'
assert healthy_export['is_healthy'] is True
assert healthy_export['manifest_hash_matches_registry'] is True
assert healthy_export['manifest_hash_matches_content'] is True
assert healthy_export['manifest_hash'] == healthy_export['computed_manifest_hash']
assert healthy_export['missing_required_paths'] == []
assert tampered_result.status == 'success'
assert tampered_export['health_status'] == 'manifest_hash_mismatch'
assert tampered_export['is_healthy'] is False
assert tampered_export['manifest_hash_matches_registry'] is True
assert tampered_export['manifest_hash_matches_content'] is False
assert tampered_export['manifest_hash'] != tampered_export['computed_manifest_hash']
assert 'health=healthy' in summary['healthy_notification']
assert 'health=manifest_hash_mismatch' in summary['tampered_notification']
summary


{'schema_version': 'research.production_manifest_hash_validation.v1',
 'strategy_id': 'paper_sim_replay_v1',
 'healthy_status': 'success',
 'healthy_health_status': 'healthy',
 'healthy_is_healthy': True,
 'healthy_manifest_hash_matches_registry': True,
 'healthy_manifest_hash_matches_content': True,
 'healthy_missing_required_paths': [],
 'healthy_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'healthy_computed_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'tampered_status': 'success',
 'tampered_health_status': 'manifest_hash_mismatch',
 'tampered_is_healthy': False,
 'tampered_manifest_hash_matches_registry': True,
 'tampered_manifest_hash_matches_content': False,
 'tampered_manifest_hash': '225b34eb7f25fcc12a97d0cd184b0f745f5978a25c24600f671620e7853c2bbf',
 'tampered_computed_manifest_hash': '46e1b4b612e7f68d583edb593eb9f00c273b159b76263ac5ced87f15ad65fc64',
 'healthy_notification': 'research_production_ma

## 中文结论

- 做了什么：使用 `src.core.production_manifest_content_hash` 重算 production manifest 内容 hash，并通过 `src.schedule.run_production_health_check_once` 分别验证真实 `production/production_registry.json` 和一个临时篡改副本。
- 核心指标：真实 registry 输出 `health_status=healthy`、`is_healthy=True`、`manifest_hash_matches_registry=True`、`manifest_hash_matches_content=True`、`missing_required_paths=[]`；篡改副本只改 manifest metadata 且保留原 hash，输出 `health_status=manifest_hash_mismatch`、`manifest_hash_matches_registry=True`、`manifest_hash_matches_content=False`。
- regime 差异：本实验不重新训练或重新评估 regime；它验证 production 健康检查能保护承载 regime/dashboard/backtest 结果的 manifest 证据面，防止发布后内容漂移。
- 是否晋升：否。本实验是框架健康检查增强，不改变 `paper_sim_replay_v1` 的策略晋升状态。
- 风险：hash 校验只能证明 manifest 内容与发布 hash 一致，不能替代模型效果、broker 可用性、数据 freshness 和实盘风控检查；v1 仍保持 paper/sim 边界。
- 下一步：在 dashboard 和 schedule 通知中继续显示 `manifest_hash_matches_content` 与 `computed_manifest_hash`，后续如增加 production 策略，应统一进入同一 registry health 检查。
